RNN

In [ ]:
import torch
import torch.nn as nn

# Define input parameters
input_size = 10  # Size of input features at each time step
hidden_size = 32 # Size of the hidden state
num_layers = 2   # Number of recurrent layers
batch_szie = 15
seq_len = 10

# Create an RNN layer
rnn_layer = nn.RNN(input_size, hidden_size, num_layers)

input_sequence = torch.randn(seq_len, batch_szie, input_size)

# Forward pass through the RNN
output, hidden = rnn_layer(input_sequence)

print("Output shape:", output.shape) # (sequence_length, batch_size, hidden_size)
print("Hidden state shape:", hidden.shape) # (num_layers, batch_size, hidden_size)

Output shape: torch.Size([10, 15, 32])
Hidden state shape: torch.Size([2, 15, 32])


LSTM

In [ ]:
    import torch
    import torch.nn as nn

    input_size = 10  # Dimension of input features
    hidden_size = 32 # Dimension of hidden state
    num_layers = 1   # Number of LSTM layers

    lstm_layer = nn.LSTM(input_size, hidden_size, num_layers)

    # Example input: (seq_len, batch_size, input_size)
    # If batch_first=True, then (batch_size, seq_len, input_size)
    input_data = torch.randn(5, 3, input_size) # 5 time steps, 3 samples, 10 features

    output, (h_n, c_n) = lstm_layer(input_data)

    print(output.shape)
    print(h_n.shape)
    print(c_n.shape)

    print(h_n)
    print(c_n)
    print(output[-1, :, :])

torch.Size([5, 3, 32])
torch.Size([1, 3, 32])
torch.Size([1, 3, 32])
tensor([[[ 0.1008,  0.0226,  0.0555,  0.0249, -0.0073,  0.0685,  0.1433,
           0.1344,  0.0295, -0.0018,  0.0539,  0.0015,  0.1034, -0.0260,
           0.0850,  0.0227, -0.0723, -0.1153,  0.0755, -0.0947,  0.0297,
          -0.0337,  0.0286, -0.1289,  0.1912,  0.0296, -0.1817, -0.0947,
           0.1435, -0.0695,  0.0606, -0.0029],
         [ 0.0007,  0.0133, -0.0061, -0.0212,  0.0425, -0.0041,  0.1236,
           0.0912,  0.0714,  0.1108,  0.0351, -0.0712,  0.1187,  0.0423,
          -0.0092, -0.0465,  0.0804,  0.0241,  0.1374, -0.1034, -0.0565,
           0.0376, -0.0103,  0.0818,  0.0179, -0.0541, -0.1851, -0.0176,
           0.0495,  0.1602,  0.0470, -0.1257],
         [-0.1080,  0.1022,  0.0994,  0.3285, -0.0080, -0.0833,  0.0459,
           0.1825,  0.3235, -0.1252, -0.0896,  0.0921,  0.0340,  0.0381,
           0.1227,  0.0740, -0.0189, -0.1511,  0.0060,  0.0770, -0.0080,
           0.2286, -0.1389,  0.098

In [ ]:
class LSTM1(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # last time step


model = LSTM1(input_size=1, hidden_size=32)
x = torch.randn(64, 10, 1)  # batch=64, seq_len=10, 1 feature
y = model(x)

print(y.shape)

torch.Size([64, 1])


BiRNN

In [ ]:
import torch
import torch.nn as nn

# Sample input: batch_size=2, seq_len=5, input_size=10
batch_size = 2
seq_len = 5
input_size = 10
hidden_size = 8
num_layers = 1

# Create dummy input
x = torch.randn(batch_size, seq_len, input_size)

# Define bidirectional RNN
rnn = nn.RNN(input_size=input_size,
             hidden_size=hidden_size,
             num_layers=num_layers,
             bidirectional=True,
             batch_first=True)

# Forward pass
output, hn = rnn(x)

print("Input shape:", x.shape)            # (2, 5, 10)
print("Output shape:", output.shape)      # (2, 5, 16) ← hidden_size * 2 because it's bidirectional
print("Hidden state shape:", hn.shape)    # (2, 2, 8) ← (num_directions, batch, hidden_size)


Input shape: torch.Size([2, 5, 10])
Output shape: torch.Size([2, 5, 16])
Hidden state shape: torch.Size([2, 2, 8])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BiRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(BiRNN, self).__init__()
        self.hidden_size = hidden_size

        # Forward RNN
        self.i2h_fwd = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o_fwd = nn.Linear(input_size + hidden_size, output_size)

        # Backward RNN
        self.i2h_bwd = nn.Linear(input_size + hidden_size, hidden_size)
        self.i2o_bwd = nn.Linear(input_size + hidden_size, output_size)

        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_seq):
        seq_len = input_seq.size(0)  # Assume (seq_len, input_size)

        # Initialize hidden states
        h_fwd = torch.zeros(1, self.hidden_size)
        h_bwd = torch.zeros(1, self.hidden_size)

        outputs_fwd = []
        outputs_bwd = []

        # Forward pass
        for t in range(seq_len):
            combined = torch.cat((input_seq[t].unsqueeze(0), h_fwd), 1)
            h_fwd = torch.tanh(self.i2h_fwd(combined))
            out_fwd = self.i2o_fwd(combined)
            outputs_fwd.append(out_fwd)

        # Backward pass
        for t in reversed(range(seq_len)):
            combined = torch.cat((input_seq[t].unsqueeze(0), h_bwd), 1)
            h_bwd = torch.tanh(self.i2h_bwd(combined))
            out_bwd = self.i2o_bwd(combined)
            outputs_bwd.insert(0, out_bwd)

        # Combine outputs (e.g., concatenate or sum)
        outputs = [self.softmax(f + b) for f, b in zip(outputs_fwd, outputs_bwd)]
        return outputs  # List of (1, output_size) tensors

# Example usage
seq_len = 5
input_size = 10
hidden_size = 8
output_size = 4

model = BiRNN(input_size, hidden_size, output_size)
input_seq = torch.randn(seq_len, input_size)

outputs = model(input_seq)
for t, o in enumerate(outputs):
    print(f"Time {t} output:", o)


Time 0 output: tensor([[-0.4764, -1.9323, -1.6200, -3.3175]], grad_fn=<LogSoftmaxBackward0>)
Time 1 output: tensor([[-0.7433, -2.6563, -1.4547, -1.5107]], grad_fn=<LogSoftmaxBackward0>)
Time 2 output: tensor([[-0.4875, -2.5419, -1.9005, -1.8475]], grad_fn=<LogSoftmaxBackward0>)
Time 3 output: tensor([[-1.8451, -1.1235, -0.9584, -2.0148]], grad_fn=<LogSoftmaxBackward0>)
Time 4 output: tensor([[-1.3505, -1.4406, -0.9159, -2.2640]], grad_fn=<LogSoftmaxBackward0>)


Word Embeddings

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# Toy corpus
sentences = [
    "I love NLP",
    "NLP is fun",
    "I hate bugs"
]
labels = [1, 1, 0]  # 1: positive, 0: negative


In [6]:
print(F.one_hot(torch.arange(0, 5)))

tensor([[1, 0, 0, 0, 0],
        [0, 1, 0, 0, 0],
        [0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0],
        [0, 0, 0, 0, 1]])


In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, embedding_dim, embedding_layer, hidden_dim, output_dim):
        super().__init__()
        self.embedding = embedding_layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        return self.fc(hidden.squeeze(0))


In [ ]:
class OneHotEmbedding(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

    def forward(self, x):
        return F.one_hot(x, num_classes=self.vocab_size).float()


In [8]:
import torch
import torch.nn as nn

embedding = nn.Embedding(10, 3)

input = torch.LongTensor([[1, 2, 4, 5], [4, 3, 2, 9]])
print(embedding(input))

tensor([[[-0.2402, -1.7949,  0.7654],
         [ 0.3661,  0.5188, -0.0421],
         [-0.1224, -0.0435, -0.4207],
         [-0.5193,  0.6558, -0.5512]],

        [[-0.1224, -0.0435, -0.4207],
         [-1.8701, -0.8215, -0.6524],
         [ 0.3661,  0.5188, -0.0421],
         [-0.3500,  0.9225, -0.9076]]], grad_fn=<EmbeddingBackward0>)
